In [ ]:
'''# Open loop simulation

E_grid_vec = np.zeros(Nsim)   # Wh per step
cost_vec   = np.zeros(Nsim)   # R$ per step
SoC_vec    = np.zeros(Nsim)
P_batvec   = np.zeros(Nsim)
E_curt_vec = np.zeros(Nsim)

E_load = 0.0  # Wh
E_pv   = 0.0  # Wh
E_bat  = 0.0  # Wh 


for k in range(Nsim):

    bat1.SoC, E_grid, E_curt, cost, P_bat = bat1.step_open_loop(
    SoC = bat1.SoC, P_load = load_profile[k], P_pv = pv_profile[k], tariff = tariff_sim[k], dt = dt, P_bat = 50)

    E_grid_vec[k] = E_grid
    cost_vec[k]   = cost
    P_batvec[k]   = P_bat
    SoC_vec[k]    = bat1.SoC
    E_curt_vec[k] = E_curt

    E_load += load_profile[k] * (dt) # Convert to Wh
    E_pv   += pv_profile[k] * (dt) 
    E_bat  += P_bat * (dt) 
    

cost_total = float(np.sum(cost_vec))       # R$
grid_total = np.sum(E_grid_vec)/1000.0
load_total = E_load/1000.0 
pv_total   = E_pv/1000.0
bat_total  = E_bat/1000.0
curt_total = np.sum(E_curt_vec)/1000.0


print(f'Total cost over {t_sim} hours: R${cost_total:.2f}')
print(f'Total load consumption: {load_total:.2f} kWh')
print(f'Total grid consumption over {t_sim} hours: {grid_total:.2f} kWh')
print(f'Total pv generation over {t_sim} hours: {pv_total:.2f} kWh')
print(f'Total battery contribution over {t_sim} hours: {bat_total:.2f} kWh')

#print(f'Total cost over 30 days: R${cost_total*30:.2f}')
#print(f'Total load consumption: {load_total*30:.2f} kWh')
#print(f'Total grid consumption over 30 days: {grid_total*30:.2f} kWh')
#print(f'Total pv generation over 30 days: {pv_total*30:.2f} kWh')
#print(f'Total battery contribution over 30 days: {bat_total*30:.2f} kWh')



plt.figure()
plt.plot(np.arange(0, t_sim, dt), SoC_vec)
plt.title('Battery State of Charge over Time - Open Loop')
plt.xlabel('Days')
plt.ylabel('State of Charge [%]')
plt.grid()
plt.show()



plt.figure()
plt.plot(np.arange(0, t_sim, dt), E_grid_vec/1000.0)
plt.title('Grid Energy Consumption over Time - Open Loop')
plt.xlabel('Days')
plt.ylabel('Grid Energy [kWh]')
plt.grid()
plt.show()
'''


In [ ]:
'''###### Charging Tests 

SoCk = np.zeros(Nsim)
SoCk[0] = 100 # Initial SoC


#load consumption profile
for k in range(1, Nsim):
        SoC = 0.995*SoCk[k-1] + K_ch[0]*(-load_profile[k-1] + pv_profile[k-1]) 
        SoC = np.clip(SoC, 0, 100)
        SoCk[k] = SoC

plt.figure()
plt.plot(np.arange(0, t_sim, dt), SoCk)
plt.title('Battery State of Charge over Time - Open Loop')
plt.xlabel('Time [hours]')
plt.ylabel('State of Charge [%]')
plt.grid()
plt.show()
'''

In [ ]:
########## Model
# Baterry Model
# SoC_a = previous SoC [Wh]
# loss = 0.995 == Self discharge per sample
#### SoC(k) = loss*SoC(k-1) + K1 * Pot(k-1)


A = np.array([1.0, -0.995])  # A polynomial coefficients
B = np.array([K_ch[0]])  # B polynomial coefficients 


# dt = Sampling time in hours


## GPC Matrices for Battery SoC Control

CARIMA = CARIMA(A, B, Np, Nu)
G = CARIMA["G"]
F = CARIMA["F"]
